# Final evaluation audit
This notebook reads verified saved outputs. It does not refit or reopen an untouched holdout. Install requirements-ml.lock.txt for the research environment. Run all cells in order. The 19,999-draw results are a disclosed post-exposure correction of primary p-value resolution.

In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / "reports").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "config/evaluation.json").exists(), "Run from repository root or notebooks/"
def checked_csv(folder, name):
    path = ROOT / "reports" / folder
    manifest = json.loads((path / "manifest.json").read_text())
    assert manifest["status"] == "passed"
    assert hashlib.sha256((path / name).read_bytes()).hexdigest() == manifest["artifacts"][name]
    return pd.read_csv(path / name)


In [ ]:
primary = checked_csv("phase5", "calibration_scores.csv")
paired = checked_csv("phase5-sensitivity", "paired_comparisons.csv")
assert len(primary) == 54 and len(paired) == 165
print(primary.groupby("asset").common_scored_rows.first().to_string())
print("Sensitivity adjusted comparison rejections:", int((paired.pvalue_holm < .05).sum()))

In [ ]:
weather = paired.loc[(paired.reference != "historical") & paired.metric.eq("quantile_loss")]
print(weather[["asset", "candidate", "confidence", "mean", "ci_low", "ci_high", "pvalue_holm"]].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (asset, frame) in zip(axes, primary.loc[primary.confidence.eq(.99)].groupby("asset")):
    ax.barh(frame.model, 100 * frame.exceedance_rate)
    ax.axvline(1, color="black", linestyle="--")
    ax.set_title(asset); ax.set_xlabel("99% VaR exceedance rate (%)")
fig.tight_layout()
plt.show()

To reproduce the sealed forecasts, run `make evaluate` from the root with the preserved input snapshot. Use `make sensitivity` for the separate bootstrap correction. Replays are not independent holdouts. Do not reseal after exposure.